In [1]:
pip install transformers datasets evaluate accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')
# This unzips the file into a folder called 'solar_data' inside Colab
!unzip "/content/drive/MyDrive/solar_dust.zip" -d "/content/solar_data"

Mounted at /content/drive
Archive:  /content/drive/MyDrive/solar_dust.zip
  inflating: /content/solar_data/Detect_solar_dust/Clean/Imgclean_0_0.jpg  
  inflating: /content/solar_data/Detect_solar_dust/Clean/Imgclean_1000_0.jpg  
  inflating: /content/solar_data/Detect_solar_dust/Clean/Imgclean_1001_0.jpg  
  inflating: /content/solar_data/Detect_solar_dust/Clean/Imgclean_1002_0.jpg  
  inflating: /content/solar_data/Detect_solar_dust/Clean/Imgclean_1003_0.jpg  
  inflating: /content/solar_data/Detect_solar_dust/Clean/Imgclean_1004_0.jpg  
  inflating: /content/solar_data/Detect_solar_dust/Clean/Imgclean_1005_0.jpg  
  inflating: /content/solar_data/Detect_solar_dust/Clean/Imgclean_1006_0.jpg  
  inflating: /content/solar_data/Detect_solar_dust/Clean/Imgclean_1007_0.jpg  
  inflating: /content/solar_data/Detect_solar_dust/Clean/Imgclean_1008_0.jpg  
  inflating: /content/solar_data/Detect_solar_dust/Clean/Imgclean_1009_0.jpg  
  inflating: /content/solar_data/Detect_solar_dust/Clean/Img

In [3]:
from datasets import load_dataset
from transformers import ViTImageProcessor

# 1. Load your local images
ds = load_dataset("imagefolder", data_dir="/content/solar_data")

# Immediately split it for training/testing
ds = ds["train"].train_test_split(test_size=0.2)
# 2. Load the processor (it handles resizing and normalization)
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

def transform(example_batch):
    # Add convert_rgb=True here to force images into 3 channels (RGB)
    inputs = processor(
        [x.convert("RGB") for x in example_batch['image']],
        return_tensors='pt'
    )
    inputs['labels'] = example_batch['label']
    return inputs

# Apply the transformation
prepared_ds = ds.with_transform(transform)

Resolving data files:   0%|          | 0/2562 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [4]:
from transformers import ViTForImageClassification

labels = ds['train'].features['label'].names

model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=len(labels),
    id2label={str(i): c for i, c in enumerate(labels)},
    label2id={c: str(i) for i, c in enumerate(labels)},
    ignore_mismatched_sizes=True # This allows us to swap the 1000-class head for a 2-class head
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./vit-solar-dust",
    per_device_train_batch_size=16,

    # CHANGE THIS LINE:
    eval_strategy="steps", # used to be evaluation_strategy

    num_train_epochs=3,
    save_steps=100,
    eval_steps=100,
    logging_steps=10,
    learning_rate=2e-5,
    remove_unused_columns=False,
    push_to_hub=False,
    report_to="none",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=prepared_ds["train"],
    eval_dataset=prepared_ds["train"], # In a real project, use a separate test split!
    tokenizer=processor,
    compute_metrics=compute_metrics,
)

trainer.train()

/tmp/ipython-input-2137311281.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,Accuracy
100,0.431800,0.222169,0.923377
200,0.142800,0.115577,0.965837
300,0.043200,0.064330,0.979502


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


TrainOutput(global_step=387, training_loss=0.2197494299682844, metrics={'train_runtime': 800.2915, 'train_samples_per_second': 7.681, 'train_steps_per_second': 0.484, 'total_flos': 4.76343260160897e+17, 'train_loss': 0.2197494299682844, 'epoch': 3.0})

In [6]:
# Save to a local folder in Colab
model.save_pretrained("./solar-dust-model")
processor.save_pretrained("./solar-dust-model")

# Download to computer
import shutil
shutil.make_archive("solar_model", 'zip', "./solar-dust-model")

from google.colab import files
files.download("solar_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
metrics = trainer.evaluate()
print(f"Final Accuracy: {metrics['eval_accuracy'] * 100:.2f}%")

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Final Accuracy: 97.95%
